In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder

BASE_OUTPUT_DIR = "/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs"
OUTLIER_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "outlier_processed_outputs")

DATA_PATH = os.path.join(OUTLIER_OUTPUT_DIR,
    "GlobalWeatherRepository_missing_outlier_cleaned.csv")

CORR_OUTPUT_DIR = os.path.join(OUTLIER_OUTPUT_DIR,
    "correlation_analysis_outputs")

GLOBAL_DIR = os.path.join(CORR_OUTPUT_DIR, "global")
CITY_DIR = os.path.join(CORR_OUTPUT_DIR, "representative_cities")
FIG_DIR = os.path.join(CORR_OUTPUT_DIR, "figures")
REPORT_DIR = os.path.join(CORR_OUTPUT_DIR, "reports")

for p in [CORR_OUTPUT_DIR, GLOBAL_DIR, CITY_DIR, FIG_DIR, REPORT_DIR]:
    os.makedirs(p, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")
df = df.dropna(subset=["location_name", "last_updated"])
df = df.sort_values(["location_name", "last_updated"]).reset_index(drop=True)

print("Loaded data:", df.shape)
display(df.head())

Mounted at /content/drive
Loaded data: (141508, 41)


,last_updated,country,location_name,latitude,longitude,timezone,last_updated_epoch,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,2024-05-31 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717164900,16.0,60.8,Moderate rain,...,2.2,3.4,1,1,05:36 AM,09:52 PM,02:59 AM,02:09 PM,Waning Crescent,47
1,2024-06-01 16:30:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717252200,16.0,60.8,Overcast,...,6.5,19.6,1,1,05:35 AM,09:53 PM,03:12 AM,03:34 PM,Waning Crescent,35
2,2024-06-04 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717510500,18.0,64.4,Partly cloudy,...,5.7,6.0,1,1,05:33 AM,09:56 PM,03:56 AM,07:57 PM,Waning Crescent,8
3,2024-06-05 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717596900,15.0,59.0,Partly cloudy,...,1.2,2.4,1,1,05:33 AM,09:56 PM,04:17 AM,09:25 PM,Waning Crescent,3
4,2024-06-11 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1718115300,15.2,59.4,Partly cloudy,...,0.5,0.9,1,1,05:30 AM,10:01 PM,10:16 AM,01:31 AM,Waxing Crescent,21


In [2]:
# Define cities and correlation variables
selected_cities = [
    "Tokyo",
    "Baghdad",
    "Bern",
    "Suva",
    "Dakar",
    "Kyiv",
    "Accra",
    "Kabul",
    "Valletta",
    "Warsaw"
]

exclude_numeric_cols = [
    "last_updated_epoch",
    "latitude",
    "longitude"
]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

candidate_corr_vars = [
    c for c in numeric_cols
    if c not in exclude_numeric_cols
]

# Keep only variables that actually vary over time
corr_vars = []
for col in candidate_corr_vars:
    if df[col].nunique(dropna=True) > 1:
        corr_vars.append(col)

print("Variables used for correlation analysis:")
print(corr_vars)
print("Number of variables:", len(corr_vars))

# encode condition_text for separate MI-style categorical analysis
if "condition_text" in df.columns:
    df["condition_text_encoded"] = LabelEncoder().fit_transform(
        df["condition_text"].astype(str))
    condition_mapping = (
        df[["condition_text", "condition_text_encoded"]]
        .drop_duplicates()
        .sort_values("condition_text_encoded"))
    condition_mapping.to_csv(
        os.path.join(REPORT_DIR, "condition_text_label_encoding_mapping.csv"),
        index=False)
    print("condition_text was label-encoded and mapping was saved.")

Variables used for correlation analysis:
['temperature_celsius', 'temperature_fahrenheit', 'wind_mph', 'wind_kph', 'wind_degree', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'moon_illumination']
Number of variables: 27
condition_text was label-encoded and mapping was saved.


In [3]:
# Correlation / dependence functions

def clean_numeric_matrix(data, variables):
    x = data[variables].copy()
    for col in variables:
        x[col] = pd.to_numeric(x[col], errors="coerce")
    x = x.replace([np.inf, -np.inf], np.nan)
    x = x.dropna(axis=0)
    return x

def pearson_corr_matrix(data, variables):
    x = clean_numeric_matrix(data, variables)
    return x.corr(method="pearson")

def spearman_corr_matrix(data, variables):
    x = clean_numeric_matrix(data, variables)
    return x.corr(method="spearman")

def mutual_information_matrix(data, variables, random_state=42):
    x = clean_numeric_matrix(data, variables)
    mi_mat = pd.DataFrame(
        np.zeros((len(variables), len(variables))),
        index=variables,
        columns=variables)
    if len(x) < 5:
        return mi_mat
    for target in variables:
        y = x[target].values
        for source in variables:
            if source == target:
                mi_mat.loc[source, target] = 1.0
            else:
                X = x[[source]].values
                try:
                    mi = mutual_info_regression(X,y,
                        random_state=random_state)[0]
                except Exception:
                    mi = np.nan
                mi_mat.loc[source, target] = mi
    return mi_mat

def distance_correlation_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    n = len(x)
    if n < 5:
        return np.nan
    x = x.reshape(-1, 1)
    y = y.reshape(-1, 1)
    a = np.abs(x - x.T)
    b = np.abs(y - y.T)
    A = a - a.mean(axis=0, keepdims=True) - a.mean(axis=1, keepdims=True) + a.mean()
    B = b - b.mean(axis=0, keepdims=True) - b.mean(axis=1, keepdims=True) + b.mean()
    dcov2 = np.mean(A * B)
    dvarx2 = np.mean(A * A)
    dvary2 = np.mean(B * B)
    if dvarx2 <= 1e-12 or dvary2 <= 1e-12:
        return 0.0
    return np.sqrt(max(dcov2, 0)) / np.sqrt(np.sqrt(dvarx2 * dvary2))

def distance_correlation_matrix(data, variables, max_rows=5000):
    x = clean_numeric_matrix(data, variables)
    if len(x) > max_rows:
        x = x.sample(max_rows, random_state=42)
    dcor_mat = pd.DataFrame(
        np.zeros((len(variables), len(variables))),
        index=variables,
        columns=variables)
    for i, v1 in enumerate(variables):
        for j, v2 in enumerate(variables):
            if i == j:
                dcor_mat.loc[v1, v2] = 1.0
            elif j < i:
                dcor_mat.loc[v1, v2] = dcor_mat.loc[v2, v1]
            else:
                dcor_mat.loc[v1, v2] = distance_correlation_1d(
                    x[v1].values,
                    x[v2].values)
    return dcor_mat

def max_abs_cross_correlation_pair(x, y, max_lag=30):
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)
    valid = x.notna() & y.notna()
    x = x[valid]
    y = y[valid]
    if len(x) < max_lag + 5:
        return np.nan, np.nan
    best_corr = np.nan
    best_lag = np.nan
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            x_lag = x.iloc[:lag]
            y_lag = y.iloc[-lag:]
        elif lag > 0:
            x_lag = x.iloc[lag:]
            y_lag = y.iloc[:-lag]
        else:
            x_lag = x
            y_lag = y

        if len(x_lag) < 5:
            continue
        corr = np.corrcoef(x_lag, y_lag)[0, 1]
        if np.isnan(corr):
            continue
        if np.isnan(best_corr) or abs(corr) > abs(best_corr):
            best_corr = corr
            best_lag = lag
    return best_corr, best_lag

def cross_correlation_matrix_single_series(data, variables, max_lag=30):
    x = clean_numeric_matrix(data, variables)
    corr_mat = pd.DataFrame(
        np.zeros((len(variables), len(variables))),
        index=variables,
        columns=variables)
    lag_mat = pd.DataFrame(
        np.zeros((len(variables), len(variables))),
        index=variables,
        columns=variables)
    for v1 in variables:
        for v2 in variables:
            if v1 == v2:
                corr_mat.loc[v1, v2] = 1.0
                lag_mat.loc[v1, v2] = 0
            else:
                corr, lag = max_abs_cross_correlation_pair(
                    x[v1].values,
                    x[v2].values,
                    max_lag=max_lag)
                corr_mat.loc[v1, v2] = corr
                lag_mat.loc[v1, v2] = lag
    return corr_mat, lag_mat

def global_citywise_cross_correlation(data, variables, max_lag=30):
    city_corrs = []
    city_lags = []
    for city, g in data.groupby("location_name"):
        if len(g) < max_lag + 10:
            continue
        g = g.sort_values("last_updated")
        try:
            corr_mat, lag_mat = cross_correlation_matrix_single_series(
                g,
                variables,
                max_lag=max_lag)
            city_corrs.append(corr_mat)
            city_lags.append(lag_mat)
        except Exception:
            continue
    mean_corr = sum(city_corrs) / len(city_corrs)
    mean_lag = sum(city_lags) / len(city_lags)
    return mean_corr, mean_lag

In [4]:
# Heatmap helper

def plot_heatmap(matrix, title, save_path, figsize=(12, 10)):
    plt.figure(figsize=figsize)
    plt.imshow(matrix.values, aspect="auto")
    plt.colorbar(label="Value")
    plt.xticks(
        ticks=np.arange(len(matrix.columns)),
        labels=matrix.columns,
        rotation=90)
    plt.yticks(
        ticks=np.arange(len(matrix.index)),
        labels=matrix.index)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def save_matrix_and_heatmap(matrix, out_dir, fig_dir, name, title):
    csv_path = os.path.join(out_dir, f"{name}.csv")
    fig_path = os.path.join(fig_dir, f"{name}.png")
    matrix.to_csv(csv_path)
    plot_heatmap(matrix, title, fig_path)
    return csv_path, fig_path

In [5]:
# Global correlation analysis

GLOBAL_FIG_DIR = os.path.join(FIG_DIR, "global")
os.makedirs(GLOBAL_FIG_DIR, exist_ok=True)

global_df = df.copy()

print("Running global Pearson...")
global_pearson = pearson_corr_matrix(global_df, corr_vars)

print("Running global Spearman...")
global_spearman = spearman_corr_matrix(global_df, corr_vars)

print("Running global Mutual Information...")
global_mi = mutual_information_matrix(global_df, corr_vars)

print("Running global Distance Correlation...")
global_dcor = distance_correlation_matrix(global_df, corr_vars, max_rows=5000)

print("Running global city-wise averaged Cross-correlation...")
global_xcorr, global_xcorr_lag = global_citywise_cross_correlation(
    global_df, corr_vars, max_lag=30)

save_matrix_and_heatmap(global_pearson, GLOBAL_DIR, GLOBAL_FIG_DIR,
    "global_pearson_correlation", "Global Pearson Correlation")

save_matrix_and_heatmap(global_spearman, GLOBAL_DIR, GLOBAL_FIG_DIR,
    "global_spearman_correlation", "Global Spearman Correlation")

save_matrix_and_heatmap(global_mi,GLOBAL_DIR,GLOBAL_FIG_DIR,
    "global_mutual_information","Global Mutual Information")

save_matrix_and_heatmap(global_dcor,GLOBAL_DIR,GLOBAL_FIG_DIR,
    "global_distance_correlation","Global Distance Correlation")

save_matrix_and_heatmap(global_xcorr,GLOBAL_DIR,GLOBAL_FIG_DIR,
    "global_cross_correlation_max_abs",
    "Global City-wise Averaged Max Absolute Cross-correlation")

save_matrix_and_heatmap(global_xcorr_lag,GLOBAL_DIR,GLOBAL_FIG_DIR,
    "global_cross_correlation_best_lag","Global City-wise Averaged Best Lag")

print("Global correlation analysis completed.")

Running global Pearson...
Running global Spearman...
Running global Mutual Information...
Running global Distance Correlation...
Running global city-wise averaged Cross-correlation...
Global correlation analysis completed.


In [6]:
# Representative city correlation analysis

CITY_FIG_DIR = os.path.join(FIG_DIR, "representative_cities")
os.makedirs(CITY_FIG_DIR, exist_ok=True)

city_summary_records = []

for city in selected_cities:
    print(f"Running city-level correlation analysis: {city}")
    city_df = df[df["location_name"] == city].copy()
    city_df = city_df.sort_values("last_updated")
    if len(city_df) < 50:
        print(f"Skipped {city}: insufficient records.")
        continue
    city_safe = str(city).replace(" ", "_").replace("/", "_").replace("\\", "_")
    city_out_dir = os.path.join(CITY_DIR, city_safe)
    city_fig_dir = os.path.join(CITY_FIG_DIR, city_safe)
    os.makedirs(city_out_dir, exist_ok=True)
    os.makedirs(city_fig_dir, exist_ok=True)
    city_corr_vars = [col for col in corr_vars
        if city_df[col].nunique(dropna=True) > 1]
    pearson = pearson_corr_matrix(city_df, city_corr_vars)
    spearman = spearman_corr_matrix(city_df, city_corr_vars)
    mi = mutual_information_matrix(city_df, city_corr_vars)
    dcor = distance_correlation_matrix(city_df, city_corr_vars, max_rows=5000)

    xcorr, xcorr_lag = cross_correlation_matrix_single_series(
        city_df,city_corr_vars,max_lag=30)

    save_matrix_and_heatmap(pearson,city_out_dir,city_fig_dir,
        f"{city_safe}_pearson_correlation",
        f"{city} Pearson Correlation")

    save_matrix_and_heatmap(spearman,city_out_dir,city_fig_dir,
        f"{city_safe}_spearman_correlation",
        f"{city} Spearman Correlation")

    save_matrix_and_heatmap(mi,city_out_dir,city_fig_dir,
        f"{city_safe}_mutual_information",
        f"{city} Mutual Information")

    save_matrix_and_heatmap(dcor,city_out_dir,city_fig_dir,
        f"{city_safe}_distance_correlation",
        f"{city} Distance Correlation")

    save_matrix_and_heatmap(xcorr,city_out_dir,city_fig_dir,
        f"{city_safe}_cross_correlation_max_abs",
        f"{city} Max Absolute Cross-correlation")

    save_matrix_and_heatmap(xcorr_lag,city_out_dir,city_fig_dir,
        f"{city_safe}_cross_correlation_best_lag",
        f"{city} Best Cross-correlation Lag")

    city_summary_records.append({"city": city,
        "records": len(city_df),
        "variables_used": len(city_corr_vars)})

city_corr_summary = pd.DataFrame(city_summary_records)

city_corr_summary.to_csv(
    os.path.join(REPORT_DIR, "representative_city_correlation_summary.csv"),
    index=False)

display(city_corr_summary)
print("Representative city correlation analysis completed.")

Running city-level correlation analysis: Tokyo
Running city-level correlation analysis: Baghdad
Running city-level correlation analysis: Bern
Running city-level correlation analysis: Suva
Running city-level correlation analysis: Dakar
Running city-level correlation analysis: Kyiv
Running city-level correlation analysis: Accra
Running city-level correlation analysis: Kabul
Running city-level correlation analysis: Valletta
Running city-level correlation analysis: Warsaw


,city,records,variables_used
0,Tokyo,728,27
1,Baghdad,728,27
2,Bern,728,27
3,Suva,728,27
4,Dakar,728,27
5,Kyiv,728,27
6,Accra,728,27
7,Kabul,728,27
8,Valletta,728,27
9,Warsaw,728,27


Representative city correlation analysis completed.


In [7]:
# Extract important pairwise relationships

important_pairs = [
    ("temperature_celsius", "humidity"),
    ("temperature_celsius", "pressure_mb"),
    ("temperature_celsius", "air_quality_PM2.5"),
    ("humidity", "precip_mm"),
    ("wind_kph", "air_quality_PM2.5"),
    ("wind_kph", "air_quality_PM10"),
    ("visibility_km", "air_quality_PM2.5"),
    ("visibility_km", "air_quality_PM10"),
    ("air_quality_PM2.5", "air_quality_PM10")]

pair_records = []
def get_pair_value(mat, a, b):
    if a in mat.index and b in mat.columns:
        return mat.loc[a, b]
    return np.nan

# Global important pairs
for a, b in important_pairs:
    if a in corr_vars and b in corr_vars:
        pair_records.append({
            "scope": "global",
            "city": "ALL",
            "var_1": a,
            "var_2": b,
            "pearson": get_pair_value(global_pearson, a, b),
            "spearman": get_pair_value(global_spearman, a, b),
            "mutual_information": get_pair_value(global_mi, a, b),
            "distance_correlation": get_pair_value(global_dcor, a, b),
            "cross_correlation": get_pair_value(global_xcorr, a, b),
            "best_lag": get_pair_value(global_xcorr_lag, a, b)})

# City important pairs
for city in selected_cities:
    city_safe = str(city).replace(" ", "_").replace("/", "_").replace("\\", "_")
    city_out_dir = os.path.join(CITY_DIR, city_safe)

    try:
        pearson = pd.read_csv(
            os.path.join(city_out_dir, f"{city_safe}_pearson_correlation.csv"),
            index_col=0)
        spearman = pd.read_csv(
            os.path.join(city_out_dir, f"{city_safe}_spearman_correlation.csv"),
            index_col=0)
        mi = pd.read_csv(
            os.path.join(city_out_dir, f"{city_safe}_mutual_information.csv"),
            index_col=0)
        dcor = pd.read_csv(
            os.path.join(city_out_dir, f"{city_safe}_distance_correlation.csv"),
            index_col=0)
        xcorr = pd.read_csv(
            os.path.join(city_out_dir, f"{city_safe}_cross_correlation_max_abs.csv"),
            index_col=0)
        xlag = pd.read_csv(
            os.path.join(city_out_dir, f"{city_safe}_cross_correlation_best_lag.csv"),
            index_col=0)
    except Exception:
        continue
    for a, b in important_pairs:
        pair_records.append({
            "scope": "city",
            "city": city,
            "var_1": a,
            "var_2": b,
            "pearson": get_pair_value(pearson, a, b),
            "spearman": get_pair_value(spearman, a, b),
            "mutual_information": get_pair_value(mi, a, b),
            "distance_correlation": get_pair_value(dcor, a, b),
            "cross_correlation": get_pair_value(xcorr, a, b),
            "best_lag": get_pair_value(xlag, a, b)})

pair_summary = pd.DataFrame(pair_records)
pair_summary.to_csv(
    os.path.join(REPORT_DIR, "important_pairwise_correlation_summary.csv"),
    index=False)
display(pair_summary.head(30))

,scope,city,var_1,var_2,pearson,spearman,mutual_information,distance_correlation,cross_correlation,best_lag
0,global,ALL,temperature_celsius,humidity,-0.338869,-0.326126,0.823827,0.304051,-0.482256,0.789474
1,global,ALL,temperature_celsius,pressure_mb,-0.417808,-0.480009,0.270311,0.467393,-0.350520,3.162679
2,global,ALL,temperature_celsius,air_quality_PM2.5,0.060286,0.075433,0.068052,0.120510,-0.097135,3.377990
3,global,ALL,humidity,precip_mm,0.170663,0.361773,0.100565,0.242907,0.176230,0.244019
4,global,ALL,wind_kph,air_quality_PM2.5,-0.051847,-0.126372,0.071823,0.091506,-0.068961,-0.267943
5,global,ALL,wind_kph,air_quality_PM10,0.080586,-0.031080,0.067221,0.117736,-0.010669,-0.296651
6,global,ALL,visibility_km,air_quality_PM2.5,-0.116434,-0.159186,0.047087,0.155599,NaN,NaN
7,global,ALL,visibility_km,air_quality_PM10,-0.050023,-0.140818,0.043196,0.121006,NaN,NaN
8,global,ALL,air_quality_PM2.5,air_quality_PM10,0.653176,0.951470,1.933912,0.836107,0.921972,0.000000
9,city,Tokyo,temperature_celsius,humidity,0.316133,0.332529,0.455599,0.393876,0.445786,22.000000


In [8]:
# Advanced Correlation Analysis
# Transfer Entropy, DTW, Granger, Coherence, Wavelet Coherence
# For 10 representative cities

import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import coherence
from scipy.ndimage import gaussian_filter
from statsmodels.tsa.stattools import grangercausalitytests
try:
    import pywt
except ImportError:
    !pip install PyWavelets
    import pywt

# Paths and data loading

BASE_OUTPUT_DIR = "/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs"
OUTLIER_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "outlier_processed_outputs")

DATA_PATH = os.path.join(OUTLIER_OUTPUT_DIR,
    "GlobalWeatherRepository_missing_outlier_cleaned.csv")

CORR_OUTPUT_DIR = os.path.join(OUTLIER_OUTPUT_DIR,
    "correlation_analysis_outputs")

ADVANCED_DIR = os.path.join(CORR_OUTPUT_DIR,
    "deep_analysis_advanced_correlation")

ADV_DATA_DIR = os.path.join(ADVANCED_DIR, "data")
ADV_FIG_DIR = os.path.join(ADVANCED_DIR, "figures")
ADV_REPORT_DIR = os.path.join(ADVANCED_DIR, "reports")

for p in [ADVANCED_DIR, ADV_DATA_DIR, ADV_FIG_DIR, ADV_REPORT_DIR]:
    os.makedirs(p, exist_ok=True)

if "df" not in globals():
    df = pd.read_csv(DATA_PATH)
    df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

df = df.dropna(subset=["location_name", "last_updated"])
df = df.sort_values(["location_name", "last_updated"]).reset_index(drop=True)

selected_cities = ["Tokyo", "Baghdad", "Bern", "Suva", "Dakar",
          "Kyiv", "Accra", "Kabul", "Valletta", "Warsaw"]

if "corr_vars" not in globals():
    exclude_numeric_cols = ["last_updated_epoch", "latitude", "longitude"]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    corr_vars = [
        c for c in numeric_cols
        if c not in exclude_numeric_cols and df[c].nunique(dropna=True) > 1]

print("Advanced correlation variables:")
print(corr_vars)
print("Number of variables:", len(corr_vars))

Advanced correlation variables:
['temperature_celsius', 'temperature_fahrenheit', 'wind_mph', 'wind_kph', 'wind_degree', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'moon_illumination']
Number of variables: 27


In [9]:
# Helper functions

import re
import unicodedata

def safe_name(x):
    x = str(x)
    x = unicodedata.normalize("NFKD", x)
    x = x.encode("ascii", "ignore").decode("ascii")
    x = re.sub(r"[^A-Za-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x)
    x = x.strip("_")
    if x == "":
        x = "unknown"
    return x


def clean_city_numeric_data(city_df, variables):
    g = city_df.sort_values("last_updated").copy()
    for col in variables:
        g[col] = pd.to_numeric(g[col], errors="coerce")
    x = g[["last_updated"] + variables].replace([np.inf, -np.inf], np.nan)
    x = x.dropna(axis=0)
    return x


def plot_heatmap(matrix, title, save_path, figsize=(12, 10)):
    plt.figure(figsize=figsize)
    plt.imshow(matrix.values, aspect="auto")
    plt.colorbar(label="Value")
    plt.xticks(
        ticks=np.arange(len(matrix.columns)),
        labels=matrix.columns,
        rotation=90)
    plt.yticks(
        ticks=np.arange(len(matrix.index)),
        labels=matrix.index)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def zscore_array(x):
    x = np.asarray(x, dtype=float)
    std = np.nanstd(x)
    if std == 0 or np.isnan(std):
        return np.zeros_like(x)
    return (x - np.nanmean(x)) / std


def discretize_quantile(x, bins=5):
    x = pd.Series(x).astype(float)
    try:
        labels = pd.qcut(x, q=bins, labels=False, duplicates="drop")
    except Exception:
        labels = pd.cut(x, bins=bins, labels=False)
    labels = labels.fillna(method="ffill").fillna(method="bfill").fillna(0)
    return labels.astype(int).values

In [10]:
# Transfer Entropy
# TE: source -> target
# Discrete, lag-1 approximation

def transfer_entropy_pair(source, target, lag=1, bins=5):
    x = discretize_quantile(source, bins=bins)
    y = discretize_quantile(target, bins=bins)
    if len(x) <= lag + 2:
        return np.nan
    y_t = y[lag:]
    y_past = y[:-lag]
    x_past = x[:-lag]
    df_te = pd.DataFrame({
        "y_t": y_t,
        "y_past": y_past,
        "x_past": x_past})
    total = len(df_te)
    te = 0.0
    p_xyz = df_te.groupby(["y_t", "y_past", "x_past"]).size() / total
    p_yx = df_te.groupby(["y_past", "x_past"]).size() / total
    p_yy = df_te.groupby(["y_t", "y_past"]).size() / total
    p_y = df_te.groupby(["y_past"]).size() / total
    for (yt, yp, xp), p1 in p_xyz.items():
        p_cond1 = p1 / p_yx.loc[(yp, xp)]
        if (yt, yp) in p_yy.index and yp in p_y.index:
            p_cond2 = p_yy.loc[(yt, yp)] / p_y.loc[yp]
        else:
            continue
        if p_cond1 > 0 and p_cond2 > 0:
            te += p1 * np.log2(p_cond1 / p_cond2)
    return te


def transfer_entropy_matrix(data, variables, lag=1, bins=5):
    mat = pd.DataFrame(
        np.zeros((len(variables), len(variables))),
        index=variables,
        columns=variables)
    for source in variables:
        for target in variables:
            if source == target:
                mat.loc[source, target] = 0.0
            else:
                mat.loc[source, target] = transfer_entropy_pair(
                    data[source].values,
                    data[target].values,
                    lag=lag,
                    bins=bins)
    return mat

In [11]:
# DTW Distance
# Lower value means more similar temporal shape

def dtw_distance_pair(x, y, max_len=500):
    x = zscore_array(x)
    y = zscore_array(y)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    if len(x) < 5:
        return np.nan

    # Downsample for efficiency
    if len(x) > max_len:
        idx = np.linspace(0, len(x) - 1, max_len).astype(int)
        x = x[idx]
        y = y[idx]
    n, m = len(x), len(y)
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        xi = x[i - 1]
        for j in range(1, m + 1):
            cost = abs(xi - y[j - 1])
            dp[i, j] = cost + min(
                dp[i - 1, j],
                dp[i, j - 1],
                dp[i - 1, j - 1])
    return dp[n, m] / (n + m)


def dtw_distance_matrix(data, variables):
    mat = pd.DataFrame(
        np.zeros((len(variables), len(variables))),
        index=variables,
        columns=variables)
    for i, v1 in enumerate(variables):
        for j, v2 in enumerate(variables):
            if i == j:
                mat.loc[v1, v2] = 0.0
            elif j < i:
                mat.loc[v1, v2] = mat.loc[v2, v1]
            else:
                mat.loc[v1, v2] = dtw_distance_pair(
                    data[v1].values,
                    data[v2].values)
    return mat

In [12]:
# Granger Causality
# source -> target
# Matrix value = minimum p-value across lags
# Smaller p-value means stronger predictive directionality

def granger_pair(source, target, maxlag=7):
    temp = pd.DataFrame({
        "target": target,
        "source": source}).dropna()
    if len(temp) < maxlag * 5:
        return np.nan
    if temp["target"].std() == 0 or temp["source"].std() == 0:
        return np.nan
    try:
        result = grangercausalitytests(
            temp[["target", "source"]],
            maxlag=maxlag,
            verbose=False)
        p_values = [
            result[lag][0]["ssr_ftest"][1]
            for lag in range(1, maxlag + 1)]
        return np.min(p_values)
    except Exception:
        return np.nan


def granger_matrix(data, variables, maxlag=7):
    mat = pd.DataFrame(
        np.ones((len(variables), len(variables))),
        index=variables,
        columns=variables)
    for source in variables:
        for target in variables:
            if source == target:
                mat.loc[source, target] = 0.0
            else:
                mat.loc[source, target] = granger_pair(
                    data[source].values,
                    data[target].values,
                    maxlag=maxlag)
    return mat

In [13]:
# Coherence
# Frequency-domain shared periodicity
# Matrix value = mean coherence across frequencies

def coherence_pair(x, y, fs=1.0):
    x = zscore_array(x)
    y = zscore_array(y)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    if len(x) < 32:
        return np.nan, None, None
    nperseg = min(128, max(32, len(x) // 4))
    try:
        f, cxy = coherence(x,y,fs=fs,nperseg=nperseg)
        return np.nanmean(cxy), f, cxy
    except Exception:
        return np.nan, None, None


def coherence_matrix(data, variables, fs=1.0):
    mat = pd.DataFrame(
        np.zeros((len(variables), len(variables))),
        index=variables,
        columns=variables)
    for i, v1 in enumerate(variables):
        for j, v2 in enumerate(variables):
            if i == j:
                mat.loc[v1, v2] = 1.0
            elif j < i:
                mat.loc[v1, v2] = mat.loc[v2, v1]
            else:
                val, _, _ = coherence_pair(
                    data[v1].values,
                    data[v2].values,
                    fs=fs)
                mat.loc[v1, v2] = val
    return mat

In [14]:
# Approximate Wavelet Coherence
# Saved only for selected important pairs

def wavelet_coherence_pair(x, y, sampling_period=1.0, max_scale=128):
    x = zscore_array(x)
    y = zscore_array(y)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    if len(x) < 64:
        return None, None, None
    max_scale = min(max_scale, len(x) // 2)
    scales = np.arange(1, max_scale)
    Wx, freqs = pywt.cwt(x,scales,"morl",
        sampling_period=sampling_period)

    Wy, _ = pywt.cwt(y,scales,"morl",
        sampling_period=sampling_period)

    Wxy = Wx * np.conj(Wy)
    S_Wxy = gaussian_filter(np.abs(Wxy) ** 2, sigma=(2, 3))
    S_Wx = gaussian_filter(np.abs(Wx) ** 2, sigma=(2, 3))
    S_Wy = gaussian_filter(np.abs(Wy) ** 2, sigma=(2, 3))
    wcoh = S_Wxy / (S_Wx * S_Wy + 1e-12)
    wcoh = np.clip(wcoh, 0, 1)
    periods = 1 / freqs
    return wcoh, periods, np.arange(len(x))


def plot_wavelet_coherence(wcoh, periods, title, save_path):
    plt.figure(figsize=(12, 6))
    plt.imshow(
        wcoh,
        aspect="auto",
        origin="lower",
        extent=[0, wcoh.shape[1], periods.min(), periods.max()])

    plt.yscale("log")
    plt.colorbar(label="Wavelet coherence")
    plt.xlabel("Time index")
    plt.ylabel("Period, log scale")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

In [15]:
# Run advanced analysis for 10 representative cities

important_wavelet_pairs = [
    ("temperature_celsius", "humidity"),
    ("temperature_celsius", "pressure_mb"),
    ("temperature_celsius", "air_quality_PM2.5"),
    ("humidity", "precip_mm"),
    ("wind_kph", "air_quality_PM2.5"),
    ("wind_kph", "air_quality_PM10"),
    ("visibility_km", "air_quality_PM2.5"),
    ("visibility_km", "air_quality_PM10"),
    ("air_quality_PM2.5", "air_quality_PM10")]

advanced_summary_records = []

for city in selected_cities:
    print(f"Running advanced correlation analysis for {city}")
    city_df = df[df["location_name"] == city].copy()
    city_df = city_df.sort_values("last_updated")
    if len(city_df) < 80:
        print(f"Skipped {city}: insufficient records.")
        continue
    city_vars = [
        v for v in corr_vars
        if v in city_df.columns and city_df[v].nunique(dropna=True) > 1]
    city_data = clean_city_numeric_data(city_df, city_vars)
    if len(city_data) < 80:
        print(f"Skipped {city}: insufficient clean rows.")
        continue
    city_safe = safe_name(city)
    city_data_dir = os.path.join(ADV_DATA_DIR, city_safe)
    city_fig_dir = os.path.join(ADV_FIG_DIR, city_safe)
    city_wcoh_dir = os.path.join(city_fig_dir, "wavelet_coherence")
    for p in [city_data_dir, city_fig_dir, city_wcoh_dir]:
        os.makedirs(p, exist_ok=True)

    # Transfer Entropy
    print("  Transfer Entropy...")
    te_mat = transfer_entropy_matrix(city_data,city_vars,lag=1,bins=5)
    te_mat.to_csv(os.path.join(city_data_dir, f"{city_safe}_transfer_entropy.csv"))
    plot_heatmap(te_mat,f"{city} Transfer Entropy",
        os.path.join(city_fig_dir, f"{city_safe}_transfer_entropy_heatmap.png"))

    # DTW
    print(" DTW Distance...")
    dtw_mat = dtw_distance_matrix(city_data,city_vars)

    dtw_mat.to_csv(os.path.join(city_data_dir, f"{city_safe}_dtw_distance.csv"))

    plot_heatmap(dtw_mat,f"{city} DTW Distance",
        os.path.join(city_fig_dir, f"{city_safe}_dtw_distance_heatmap.png"))

    # Granger
    print("  Granger Causality...")
    granger_mat = granger_matrix(city_data,city_vars,maxlag=7)
    granger_mat.to_csv(os.path.join(city_data_dir, f"{city_safe}_granger_min_pvalue.csv"))

    # For heatmap, use -log10(p) so stronger causality appears brighter
    granger_score = -np.log10(granger_mat.replace(0, 1e-12))
    granger_score = granger_score.replace([np.inf, -np.inf], np.nan)
    granger_score.to_csv(os.path.join(city_data_dir, f"{city_safe}_granger_neglog10_pvalue.csv"))
    plot_heatmap(granger_score,f"{city} Granger Causality (-log10 p-value)",
        os.path.join(city_fig_dir, f"{city_safe}_granger_neglog10_pvalue_heatmap.png"))

    # Coherence
    print(" Coherence...")
    coh_mat = coherence_matrix(city_data,city_vars,fs=1.0)
    coh_mat.to_csv(os.path.join(city_data_dir, f"{city_safe}_coherence_mean.csv"))
    plot_heatmap(coh_mat,f"{city} Mean Coherence",
        os.path.join(city_fig_dir, f"{city_safe}_coherence_mean_heatmap.png"))

    # Wavelet Coherence for selected important pairs
    print(" Wavelet Coherence for selected pairs...")
    wcoh_summary_records = []
    for var1, var2 in important_wavelet_pairs:
        if var1 not in city_vars or var2 not in city_vars:
            continue
        wcoh, periods, time_idx = wavelet_coherence_pair(
            city_data[var1].values,
            city_data[var2].values,
            sampling_period=1.0,
            max_scale=128)
        if wcoh is None:
            continue
        pair_name = f"{safe_name(var1)}__{safe_name(var2)}"
        wcoh_df = pd.DataFrame(wcoh,index=periods)
        wcoh_df.index.name = "period_index"
        wcoh_df.to_csv(os.path.join(city_data_dir,
            f"{city_safe}_{pair_name}_wavelet_coherence_matrix.csv"))
        fig_path = os.path.join(city_wcoh_dir,
            f"{city_safe}_{pair_name}_wavelet_coherence.png")
        plot_wavelet_coherence(wcoh,periods,
            f"{city} Wavelet Coherence | {var1} vs {var2}",fig_path)
        wcoh_summary_records.append({
            "city": city,
            "var1": var1,
            "var2": var2,
            "mean_wavelet_coherence": np.nanmean(wcoh),
            "max_wavelet_coherence": np.nanmax(wcoh),
            "figure_path": fig_path})

    wcoh_summary_df = pd.DataFrame(wcoh_summary_records)
    wcoh_summary_df.to_csv(
        os.path.join(city_data_dir, f"{city_safe}_wavelet_coherence_summary.csv"),
        index=False)

    advanced_summary_records.append({
        "city": city,
        "records_used": len(city_data),
        "variables_used": len(city_vars),
        "transfer_entropy_saved": True,
        "dtw_saved": True,
        "granger_saved": True,
        "coherence_saved": True,
        "wavelet_coherence_pairs": len(wcoh_summary_records)})
advanced_summary = pd.DataFrame(advanced_summary_records)
advanced_summary.to_csv(
    os.path.join(ADV_REPORT_DIR, "advanced_correlation_summary.csv"),
    index=False)

display(advanced_summary)
print("Advanced correlation analysis completed.")
print("Saved to:")
print(ADVANCED_DIR)

Running advanced correlation analysis for Tokyo
  Transfer Entropy...
 DTW Distance...
  Granger Causality...
 Coherence...
 Wavelet Coherence for selected pairs...
Running advanced correlation analysis for Baghdad
  Transfer Entropy...
 DTW Distance...
  Granger Causality...
 Coherence...
 Wavelet Coherence for selected pairs...
Running advanced correlation analysis for Bern
  Transfer Entropy...
 DTW Distance...
  Granger Causality...
 Coherence...
 Wavelet Coherence for selected pairs...
Running advanced correlation analysis for Suva
  Transfer Entropy...
 DTW Distance...
  Granger Causality...
 Coherence...
 Wavelet Coherence for selected pairs...
Running advanced correlation analysis for Dakar
  Transfer Entropy...
 DTW Distance...
  Granger Causality...
 Coherence...
 Wavelet Coherence for selected pairs...
Running advanced correlation analysis for Kyiv
  Transfer Entropy...
 DTW Distance...
  Granger Causality...
 Coherence...
 Wavelet Coherence for selected pairs...
Running adv

,city,records_used,variables_used,transfer_entropy_saved,dtw_saved,granger_saved,coherence_saved,wavelet_coherence_pairs
0,Tokyo,728,27,True,True,True,True,9
1,Baghdad,728,27,True,True,True,True,9
2,Bern,728,27,True,True,True,True,9
3,Suva,728,27,True,True,True,True,9
4,Dakar,728,27,True,True,True,True,9
5,Kyiv,728,27,True,True,True,True,9
6,Accra,728,27,True,True,True,True,9
7,Kabul,728,27,True,True,True,True,9
8,Valletta,728,27,True,True,True,True,9
9,Warsaw,728,27,True,True,True,True,9


Advanced correlation analysis completed.
Saved to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/correlation_analysis_outputs/deep_analysis_advanced_correlation


In [16]:
# Extract top relationships from advanced metrics

top_records = []

for city in selected_cities:
    city_safe = safe_name(city)
    city_data_dir = os.path.join(ADV_DATA_DIR, city_safe)
    if not os.path.exists(city_data_dir):
        continue
    files = {"transfer_entropy": f"{city_safe}_transfer_entropy.csv",
        "dtw_distance": f"{city_safe}_dtw_distance.csv",
        "granger_score": f"{city_safe}_granger_neglog10_pvalue.csv",
        "coherence": f"{city_safe}_coherence_mean.csv"}

    for method, fname in files.items():
        path = os.path.join(city_data_dir, fname)
        if not os.path.exists(path):
            continue
        mat = pd.read_csv(path, index_col=0)
        for v1 in mat.index:
            for v2 in mat.columns:
                if v1 == v2:
                    continue
                val = mat.loc[v1, v2]
                if pd.isna(val):
                    continue
                top_records.append({
                    "city": city,
                    "method": method,
                    "var1": v1,
                    "var2": v2,
                    "value": val})

top_df = pd.DataFrame(top_records)

# For DTW smaller is stronger; for others larger is stronger.
top_df["rank_value"] = top_df.apply(
    lambda r: -r["value"] if r["method"] == "dtw_distance" else r["value"],
    axis=1)

top_summary = (top_df
    .sort_values(["city", "method", "rank_value"], ascending=[True, True, False])
    .groupby(["city", "method"]).head(20))

top_summary.to_csv(
    os.path.join(ADV_REPORT_DIR, "top_advanced_relationships_by_city_method.csv"),
    index=False)

display(top_summary.head(40))

,city,method,var1,var2,value,rank_value
19386,Accra,coherence,gust_mph,gust_kph,0.999807,0.999807
19412,Accra,coherence,gust_kph,gust_mph,0.999807,0.999807
18954,Accra,coherence,temperature_celsius,temperature_fahrenheit,0.999698,0.999698
18980,Accra,coherence,temperature_fahrenheit,temperature_celsius,0.999698,0.999698
19008,Accra,coherence,wind_mph,wind_kph,0.999676,0.999676
19034,Accra,coherence,wind_kph,wind_mph,0.999676,0.999676
19251,Accra,coherence,feels_like_celsius,feels_like_fahrenheit,0.999675,0.999675
19277,Accra,coherence,feels_like_fahrenheit,feels_like_celsius,0.999675,0.999675
19143,Accra,coherence,precip_mm,precip_in,0.986594,0.986594
19169,Accra,coherence,precip_in,precip_mm,0.986594,0.986594
